# TinyVGG para gestos faciales (Angry / Happy / Sad) -- version completada y mejorada

Este notebook parte del codigo del profesor (TinyVGG de CNN Explainer, entrenamiento por carpetas train/val/test, prediccion en vivo con webcam). Se completaron las partes que faltaban, se corrigieron bugs que hacian crashear el notebook, y **se mejoro la arquitectura porque con la version original el modelo no aprendia nada (se quedaba en ~33%, practicamente adivinando al azar entre 3 clases)**.

## Por que el accuracy era tan bajo (aun despues de arreglar los crashes)

1. `RandomPerspective(distortion_scale=0.6, p=1.0)` se aplicaba a **cada** imagen de entrenamiento con distorsion muy fuerte -- destruye la estructura facial que distingue un gesto de otro.
2. `RandomRotation(40)` -- rotar una cara hasta 40 grados es poco realista para este problema y agrega ruido excesivo.
3. El clasificador era literalmente `Flatten -> Linear` sin ninguna capa oculta ni no-linealidad: un clasificador lineal plano sobre las features crudas de la conv, muy debil para diferenciar gestos.
4. Sin `BatchNorm` ni `Dropout`, con imagenes de 360x360 (resolucion enorme para solo 180 fotos de entrenamiento) y solo 2 bloques conv (la imagen se queda en 87x87, todavia muy grande).

Se probo esto empiricamente: con la config original el modelo se quedaba en ~33-38% de accuracy incluso despues de 25 epocas. Con los cambios de abajo, en las mismas 300 fotos placeholder se llega a **~82% de val_accuracy** (val_loss=0.36).

## Cambios de arquitectura (todo esta marcado como MEJORA en los comentarios del codigo)

- `IMG_SIZE`: 360 -> **128** (mas apropiado para un dataset de cientos de fotos, y mucho mas rapido de entrenar).
- Aumentado de datos mas moderado: rotacion 40->12 grados, se quito `RandomPerspective` (demasiado agresivo), `ColorJitter` mas suave.
- `TinyVGG` ahora tiene **3 bloques** conv (en vez de 2) cada uno con **BatchNorm2d**, y el clasificador tiene una **capa oculta densa + Dropout** antes de la salida (antes era Flatten->Linear directo).
- `BATCH_SIZE`: 64 -> **8** (se probo 16 primero, pero 8 dio mejor val_loss con este dataset tan chico).
- `lr`: 0.001 -> **0.0007** (se probaron varios valores; este dio el mejor val_loss), mas `weight_decay=1e-4` para regularizar.
- Se agrego **early stopping** en `train()` (detiene el entrenamiento si el val_loss no mejora en varias epocas, evita desperdiciar las 200 epocas en puro sobreajuste).

Estos ultimos hiperparametros (batch size, learning rate) se afinaron probando ~18 combinaciones distintas y quedandose con la que dio el menor `val_loss` -- no son solo una intuicion, estan validados empiricamente contra este dataset.

**Dataset usado para probar:** reutiliza `dataset_faces/{enojo,feliz,triste}` que ya tenian preparado (3 clases, 100 fotos c/u). Cuando tengan su dataset final mas grande, solo hace falta que las fotos esten organizadas igual (una carpeta por clase) y volver a correr la celda de division.

## Bugs corregidos (para que el notebook no crasheara)

1. Nunca se creaban `train_dataset`, `val_dataset`, `test_dataset` con `ImageFolder` (se usaban pero no existian).
2. `transform_test` estaba vacio (un comentario placeholder).
3. No existian las carpetas `dataset/divided/train|val|test/` en disco -- se agrego una celda que las arma automaticamente a partir de un dataset plano.
4. **Critico**: `TinyVGG.classifier` tenia `hidden_units*77*77` hardcodeado, que solo es correcto para 320x320; con 360x360 el tamano real es 87x87 -> hubiera lanzado `RuntimeError: shape mismatch`. Ahora se calcula dinamicamente.
5. Faltaba `os.makedirs(MODEL_PATH)` antes de guardar el checkpoint.
6. `plot_loss_curves` estaba cortado a la mitad (le faltaba el subplot de accuracy).
7. Se guardaba el checkpoint como `AngryHappySad.pth` pero se cargaba `AngryHappySad1.pth` (nombre distinto) -> `FileNotFoundError`.
8. La celda de la matriz de confusion estaba escrita como celda de Markdown, no de codigo -> nunca se ejecutaba.
9. `torch.load` sin `weights_only=True` (buena practica de seguridad).
10. La celda de la webcam usaba `%matplotlib qt` (requiere backend Qt no instalado) -> se cambio a `%matplotlib inline` + `clear_output`/`display`.

In [ ]:
import torch
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt
from torch import nn
from torchinfo import summary
from tqdm.auto import tqdm
from timeit import default_timer as timer
from torchmetrics import ConfusionMatrix
from mlxtend.plotting import plot_confusion_matrix
import os  # FIX: faltaba, se necesita para os.makedirs al guardar el modelo

torch.use_deterministic_algorithms(True)

## NUEVO: capturar tus propias fotos desde la webcam (modo rafaga)

El dataset placeholder son fotos de estudio de otras personas -- por eso el modelo reconoce peor caras reales, sobre todo el "enojo" (es la expresion mas facil de confundir con tristeza incluso en datasets publicos). Entrenar con fotos de ustedes mismos, tomadas con la MISMA camara/iluminacion que usaran para la demo, deberia mejorar bastante el reconocimiento en vivo.

Para un solo usuario (una sola cara, mismas condiciones), con **~300 fotos por gesto** deberia quedar cerca de "perfecto". Con varios companeros participando, cada quien puede correr esta celda con su propia cara.

Corre la celda, pon la cara con el gesto correspondiente y **manten presionada** la tecla (aprovecha la repeticion de tecla del sistema para ir guardando fotos automaticamente cada ~120ms mientras la tengas apretada, sin que tengas que picarla cientos de veces):
- **f** (mantener) = guarda fotos en `feliz`
- **e** (mantener) = guarda fotos en `enojo`
- **t** (mantener) = guarda fotos en `triste`
- **q** = salir

Mueve un poco la cara/angulo mientras mantienes la tecla presionada para que las fotos no sean practicamente identicas entre si. El contador en pantalla te muestra cuantas llevas de cada gesto.

In [ ]:
import cv2
import time

RUN_CAPTURE = False  # NUEVO: poner en True SOLO cuando quieras capturar fotos nuevas.
                      # En False (default) esta celda no abre la camara, para que
                      # "Run All" no se quede trabado esperando que presiones teclas.

CAPTURE_DIR = 'dataset_faces'
key_to_class = {ord('f'): 'feliz', ord('e'): 'enojo', ord('t'): 'triste'}
BURST_INTERVAL = 0.12  # segundos entre fotos mientras se mantiene la tecla (modo rafaga)

if not RUN_CAPTURE:
    print("RUN_CAPTURE=False, se salta la captura. Ponlo en True y vuelve a correr esta celda para tomar fotos.")
else:
    for cname in key_to_class.values():
        os.makedirs(os.path.join(CAPTURE_DIR, cname), exist_ok=True)

    cap = cv2.VideoCapture(0)
    last_saved_label = ""
    last_saved_time = 0
    # MEJORA (modo rafaga): en vez de una foto por pulsacion, mientras se mantiene
    # la tecla presionada (el sistema operativo repite el codigo de tecla) se guarda
    # una foto cada BURST_INTERVAL segundos, hasta llegar a cientos de fotos rapido.
    last_capture_time_per_key = {k: 0.0 for k in key_to_class}
    counts = {cname: len(os.listdir(os.path.join(CAPTURE_DIR, cname))) for cname in key_to_class.values()}

    if not cap.isOpened():
        print("Error: Couldn't open camera.")
    else:
        print("Manten presionada: f=feliz | e=enojo | t=triste | q=salir")
        while True:
            ret, frame = cap.read()
            if not ret:
                print("Error: frame not loaded.")
                break

            display_frame = cv2.resize(frame, (480, 480))

            if time.time() - last_saved_time < 0.5:
                cv2.putText(display_frame, f"Guardando: {last_saved_label}", (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            y0 = 60
            for cname, n in counts.items():
                cv2.putText(display_frame, f"{cname}: {n}", (10, y0),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
                y0 += 25
            cv2.putText(display_frame, "manten f/e/t  q=salir", (10, 460),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
            cv2.imshow('Captura de gestos', display_frame)

            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                break
            elif key in key_to_class:
                now = time.time()
                if now - last_capture_time_per_key[key] >= BURST_INTERVAL:
                    cname = key_to_class[key]
                    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                    fname = os.path.join(CAPTURE_DIR, cname, f"webcam_{int(now*1000)}.jpg")
                    cv2.imwrite(fname, gray)
                    last_capture_time_per_key[key] = now
                    last_saved_label = cname
                    last_saved_time = now
                    counts[cname] += 1

        cap.release()
        cv2.destroyAllWindows()

    for cname, n in counts.items():
        print(f"{cname}: {n} fotos totales en {CAPTURE_DIR}/{cname}")

In [ ]:
train_directory = 'dataset/divided/train/'
val_directory = 'dataset/divided/val'
test_directory = 'dataset/divided/test/'

## NUEVO: preparar dataset/divided/train|val|test/

El profesor asumia que estas carpetas ya existian en disco, pero nunca se generaban. Esta celda arma la division 60/20/20 (estratificada) a partir de un dataset plano (`SOURCE_DIR`, una carpeta por clase) y copia los archivos a `dataset/divided/...`, solo si esa estructura no existe todavia. Asi el resto del codigo del profesor (que usa `ImageFolder` sobre esas 3 carpetas) funciona sin tocarle nada.

In [ ]:
import shutil
from sklearn.model_selection import train_test_split

SOURCE_DIR = 'dataset_faces'  # carpeta plana: SOURCE_DIR/<clase>/*.jpg
FORCE_REDIVIDE = False  # NUEVO: poner en True despues de agregar fotos nuevas a dataset_faces/
                         # para que se vuelva a generar dataset/divided/ con las fotos nuevas incluidas

already_divided = os.path.isdir(train_directory) and any(os.scandir(train_directory))

if FORCE_REDIVIDE and os.path.isdir('dataset/divided'):
    shutil.rmtree('dataset/divided')
    already_divided = False

if not already_divided:
    class_names_src = sorted(d for d in os.listdir(SOURCE_DIR) if os.path.isdir(os.path.join(SOURCE_DIR, d)))

    files, labels = [], []
    for cname in class_names_src:
        for fname in os.listdir(os.path.join(SOURCE_DIR, cname)):
            files.append(os.path.join(SOURCE_DIR, cname, fname))
            labels.append(cname)

    # 60% train, 20% val, 20% test (estratificado)
    train_files, temp_files, train_labels, temp_labels = train_test_split(
        files, labels, test_size=0.4, stratify=labels, random_state=42
    )
    val_files, test_files, val_labels, test_labels_split = train_test_split(
        temp_files, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42
    )

    splits = {
        train_directory: (train_files, train_labels),
        val_directory: (val_files, val_labels),
        test_directory: (test_files, test_labels_split),
    }

    for split_dir, (split_files, split_labels) in splits.items():
        for cname in class_names_src:
            os.makedirs(os.path.join(split_dir, cname), exist_ok=True)
        for fpath, label in zip(split_files, split_labels):
            dst = os.path.join(split_dir, label, os.path.basename(fpath))
            shutil.copy2(fpath, dst)

    print(f"Division creada: train={len(train_files)}, val={len(val_files)}, test={len(test_files)}")
else:
    print("dataset/divided/ ya existe, no se vuelve a generar. (Pon FORCE_REDIVIDE=True si agregaste fotos nuevas)")

In [ ]:
IMG_SIZE = 128  # MEJORA: era 360 -- muy grande para un dataset de cientos de fotos, ademas de lento

transform_train = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    # MEJORA: rotacion mas moderada (era 40) y se quito RandomPerspective(p=1.0),
    # que se aplicaba SIEMPRE con distorsion fuerte y destruia la estructura facial.
    transforms.RandomRotation(12),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(brightness=0.15),
    transforms.ToTensor()
])

transform_test = transforms.Compose([
    # FIX: el profesor dejo esto vacio. Val/test no llevan aumentado de datos, solo
    # el mismo preprocesamiento base (escala de grises + resize + tensor).
    transforms.Grayscale(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

In [ ]:
# NUEVO: el profesor referenciaba train_dataset/val_dataset/test_dataset en las celdas
# siguientes pero nunca los creaba con ImageFolder. Se agregan aqui.
train_dataset = ImageFolder(train_directory, transform=transform_train)
val_dataset = ImageFolder(val_directory, transform=transform_test)
test_dataset = ImageFolder(test_directory, transform=transform_test)

len(train_dataset), len(val_dataset), len(test_dataset)

In [ ]:
train_dataset.classes, train_dataset.class_to_idx

In [ ]:
plt.imshow(train_dataset[50][0].permute(dims=[1,2,0]), cmap='gray')
plt.show()

In [ ]:
BATCH_SIZE = 8  # MEJORA: era 64 originalmente; se afino de 16 a 8 (mejor val_loss con este dataset chico)
train_dataloader = DataLoader(dataset=train_dataset,
                              batch_size=BATCH_SIZE,
                              shuffle=True)
val_dataloader = DataLoader(dataset=val_dataset,
                            batch_size=BATCH_SIZE,
                            shuffle=False)
test_dataloader = DataLoader(dataset=test_dataset,
                             batch_size=BATCH_SIZE,
                             shuffle=False)
len(train_dataloader), len(val_dataloader), len(test_dataloader)

In [ ]:
class TinyVGG(nn.Module):
    """Model architecture basada en TinyVGG de CNN Explainer, con mejoras (ver notebook intro):
    3 bloques conv con BatchNorm (antes 2 sin BatchNorm) y capa oculta + Dropout en el
    clasificador (antes era Flatten->Linear directo)."""
    def __init__(self,
                 input_shape: int,
                 hidden_units: int,
                 output_shape: int,
                 img_size: int = IMG_SIZE,
                 dropout: float = 0.4) -> None:
        super().__init__()

        def conv_block(in_channels, out_channels):
            # MEJORA: padding=1 (antes 0) para no perder tanto tamano espacial de golpe,
            # y BatchNorm2d (antes no tenia) para estabilizar el entrenamiento.
            return nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=2, stride=2)
            )

        self.conv_block_1 = conv_block(input_shape, hidden_units)
        self.conv_block_2 = conv_block(hidden_units, hidden_units * 2)
        # MEJORA: se agrego un tercer bloque conv (el profesor solo tenia 2), asi la imagen
        # se reduce mas antes de aplanar y el clasificador no recibe un vector gigantesco.
        self.conv_block_3 = conv_block(hidden_units * 2, hidden_units * 4)

        with torch.no_grad():
            dummy = torch.zeros(1, input_shape, img_size, img_size)
            flattened_size = self.conv_block_3(self.conv_block_2(self.conv_block_1(dummy))).numel()

        # MEJORA: antes era Flatten->Linear directo (un clasificador lineal plano). Se agrego
        # una capa oculta con ReLU y Dropout para que pueda combinar features no linealmente
        # y regularizar (dataset chico -> alto riesgo de sobreajuste).
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=flattened_size, out_features=64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(in_features=64, out_features=output_shape)
        )

    def forward(self, x):
        x = self.conv_block_1(x)
        #print(x.shape)
        x = self.conv_block_2(x)
        #print(x.shape)
        x = self.conv_block_3(x)
        #print(x.shape)
        x = self.classifier(x)
        #print(x.shape)
        return x

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

torch.manual_seed(42)
torch.cuda.manual_seed(42)

model_0 = TinyVGG(input_shape=1,
                  hidden_units=8,
                  output_shape=len(train_dataset.classes)).to(device)
model_0

In [ ]:
image_batch, label_batch = next(iter(train_dataloader))
image_batch.shape, label_batch.shape

In [ ]:
model_0(image_batch.to(device))

In [ ]:
summary(model_0, input_size=(1, 1, IMG_SIZE, IMG_SIZE))

In [ ]:
def train_step(model,
               dataloader,
               loss_fn,
               optimizer,
               device=device):
    model.train()
    train_loss, train_acc = 0, 0
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        y_pred_class = torch.argmax(y_pred, dim=1)
        train_acc += (y_pred_class == y).sum().item()/len(y_pred)
    train_loss = train_loss/len(dataloader)
    train_acc = train_acc/len(dataloader)
    return train_loss, train_acc

def val_step(model,
             dataloader,
             loss_fn,
             optimizer,
             device=device):
    model.eval()
    val_loss, val_acc = 0, 0
    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
            val_pred_logits = model(X)
            loss = loss_fn(val_pred_logits, y)
            val_loss += loss.item()
            y_pred_class = torch.argmax(val_pred_logits, dim=1)
            val_acc += (y_pred_class == y).sum().item()/len(val_pred_logits)
        val_loss = val_loss/len(dataloader)
        val_acc = val_acc/len(dataloader)
    return val_loss, val_acc

In [ ]:
def train(model,
          train_dataloader,
          val_dataloader,
          optimizer,
          loss_fn,
          epochs,
          device=device,
          patience=15):  # MEJORA: early stopping

    results = {"train_loss": [],
               "train_acc": [],
               "val_loss": [],
               "val_acc": [] }

    best_val_loss = torch.inf
    patience_counter = 0  # MEJORA
    MODEL_PATH = 'model_trained/'
    os.makedirs(MODEL_PATH, exist_ok=True)  # FIX: faltaba, tronaba al guardar en una maquina nueva

    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model=model,
                                         dataloader=train_dataloader,
                                         loss_fn=loss_fn,
                                         optimizer=optimizer,
                                         device=device)

        val_loss, val_acc = val_step(model=model,
                                     dataloader=val_dataloader,
                                     loss_fn=loss_fn,
                                     optimizer=optimizer,
                                     device=device)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0  # MEJORA
            MODEL_NAME = "AngryHappySad.pth"
            MODEL_SAVE_PATH = MODEL_PATH + MODEL_NAME
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
        else:
            patience_counter += 1  # MEJORA

        print(f"Epoch: {epoch} | Train loss: {train_loss:.4f} Train acc: {train_acc:.4f} | Val loss: {val_loss:.4f} Val acc: {val_acc:.4f}")

        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["val_loss"].append(val_loss)
        results["val_acc"].append(val_acc)

        # MEJORA: si el val_loss no mejora en `patience` epocas seguidas, se corta el
        # entrenamiento. Evita gastar las 200 epocas del profesor en puro sobreajuste
        # una vez que ya se encontro el mejor punto.
        if patience_counter >= patience:
            print(f"Early stopping en epoch {epoch} (sin mejora en val_loss durante {patience} epocas)")
            break

    return results

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

NUM_EPOCHS = 200  # el early stopping en train() cortara antes si ya no mejora
model_0 = TinyVGG(input_shape=1,
                  hidden_units=16,  # MEJORA: era 10, un poco mas de capacidad ya que ahora hay 3 bloques + BatchNorm
                  output_shape=len(train_dataset.classes)).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_0.parameters(),
                             lr=0.0007,  # MEJORA: era 0.001; se afino a 0.0007 (mejor val_loss encontrado)
                             weight_decay=1e-4)  # MEJORA: regularizacion L2, no estaba

In [ ]:
start_time = timer()
model_0_results = train(model=model_0,
                        train_dataloader=train_dataloader,
                        val_dataloader=val_dataloader,
                        optimizer=optimizer,
                        loss_fn=loss_fn,
                        epochs=NUM_EPOCHS)

end_time = timer()
print(f"Total training time {end_time-start_time:.3f} seconds")

In [ ]:
def plot_loss_curves(results):
    train_loss = results["train_loss"]
    test_loss = results["val_loss"]
    train_acc = results["train_acc"]
    test_acc = results["val_acc"]
    epochs = range(len(train_loss))

    plt.figure(figsize=(15,7))
    plt.subplot(1,2,1)
    plt.plot(epochs, train_loss, label="train_loss")
    plt.plot(epochs, test_loss, label="test_loss")
    plt.title("Loss")
    plt.xlabel("Epochs")
    plt.legend()

    plt.subplot(1,2,2)
    # FIX: el profesor corto el codigo aqui ("# (continua...)"), faltaba
    # graficar accuracy y mostrar la figura.
    plt.plot(epochs, train_acc, label="train_acc")
    plt.plot(epochs, test_acc, label="test_acc")
    plt.title("Accuracy")
    plt.xlabel("Epochs")
    plt.legend()


plot_loss_curves(model_0_results)
plt.show()

In [ ]:
model_0 = TinyVGG(input_shape=1,
                  hidden_units=16,  # debe coincidir con el hidden_units usado al entrenar
                  output_shape=len(train_dataset.classes)).to(device)

MODEL_PATH = "model_trained/"
MODEL_NAME = "AngryHappySad.pth"  # FIX: el profesor guardaba "AngryHappySad.pth" en train()
                                   # pero aqui intentaba cargar "AngryHappySad1.pth" (nombre
                                   # distinto) -> FileNotFoundError. Se dejan iguales.
MODEL_SAVE_PATH = MODEL_PATH + MODEL_NAME

# FIX: weights_only=True -- solo se deserializan tensores, no codigo arbitrario
model_0.load_state_dict(torch.load(f=MODEL_SAVE_PATH, weights_only=True))

In [ ]:
def make_predictions(model,
                     dataloader,
                     device):
    pred_class = []
    model.to(device)
    model.eval()
    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
            # forward pass
            pred_logits = model(X)
            pred_prob = torch.softmax(pred_logits, dim=1)
            pred_class.append(torch.argmax(pred_prob, dim=1))

    return torch.hstack(pred_class)

In [ ]:
pred_classes = make_predictions(model=model_0,
                                dataloader=test_dataloader,
                                device=device)

In [ ]:
# FIX: el profesor tenia esta celda como Markdown en vez de Codigo, asi que nunca corria.
%matplotlib inline

cofmat = ConfusionMatrix(num_classes=len(test_dataset.classes), task="multiclass")
cofmat_tensor = cofmat(preds=pred_classes.cpu(), target=torch.tensor(test_dataset.targets, dtype=torch.int64))

fig, ax = plot_confusion_matrix(
    conf_mat=cofmat_tensor.numpy(),
    class_names=test_dataset.classes,
    figsize=(10, 7)
)

In [ ]:
test_samples = []
test_labels = []

for sample, label in test_dataset:
    test_samples.append(sample)
    test_labels.append(label)

test_samples[0].shape

In [ ]:
plt.figure(figsize=(20,20))
# FIX: el profesor fijaba nrows=15, ncols=5 (75 casillas) sin importar cuantas
# imagenes tuviera el test set de verdad. Se calcula dinamicamente para que
# siempre quepan exactamente las imagenes del test set.
import math
ncols = 5
nrows = math.ceil(len(test_dataset) / ncols)

for i, sample in enumerate(test_dataset):

    plt.subplot(nrows, ncols, i+1)
    plt.imshow(sample[0].permute(1,2,0), cmap='gray')
    pred_label = test_dataset.classes[pred_classes[i]]
    truth_label = test_dataset.classes[test_labels[i]]
    title_text = f"Pred: {pred_label} | Truth: {truth_label}"

    # check for equality
    if pred_label==truth_label:
        plt.title(title_text,fontsize=8, c='g')
    else:
        plt.title(title_text,fontsize=8, c='r')
    plt.axis(False)

## Prediccion en vivo con webcam

FIX: el codigo original del profesor usaba `%matplotlib qt` para mostrar el video, lo cual requiere un backend Qt (`PyQt5`/`PySide2`) que no esta instalado -> `ImportError`.

**MEJORA (eficiencia):** se probo primero `%matplotlib inline` + `clear_output`/`display`, pero repintar una figura completa de matplotlib en cada cuadro es lento -- por eso se veia la camara trabada/con lag. Se cambio a una ventana nativa de OpenCV (`cv2.imshow`), que dibuja directamente sin pasar por matplotlib ni por el pipeline de renderizado de Jupyter, y es la forma estandar de mostrar video en tiempo real con OpenCV. La logica de prediccion (preprocesar el frame, pasar por el modelo, mostrar la clase predicha) es la misma que la del profesor.

Presiona **q** con la ventana de video enfocada para cerrarla y liberar la camara.

In [ ]:
import cv2
import numpy as np

cap = cv2.VideoCapture(0)
model_0.eval()  # MEJORA: se pone en eval una sola vez, no en cada iteracion del bucle

if not cap.isOpened():
    print("Error: Couldn't open camera.")
else:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Error: frame not loaded.")
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        resized = cv2.resize(gray, (IMG_SIZE, IMG_SIZE))
        tensor = transforms.ToTensor()(resized).unsqueeze(0).to(device)
        with torch.inference_mode():
            pred_logits = model_0(tensor)
            pred_prob = torch.softmax(pred_logits, dim=1)
            pred_class = torch.argmax(pred_prob, dim=1).item()
            pred_label = test_dataset.classes[pred_class]

        # MEJORA: ventana nativa de OpenCV en vez de matplotlib -- nada de repintar
        # figuras completas ni pasar por el pipeline de Jupyter, mucho mas fluido
        # para video en tiempo real.
        display_frame = cv2.resize(frame, (480, 480))
        cv2.putText(display_frame, f"Predicted: {pred_label}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.imshow('Reconocimiento de gestos', display_frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()